## 2 卷积和池化层

### 2.1 理论计算题

**题目：** 输入大小 $3\times32\times32$，卷积层包含 16 个 $3\times5\times5$ 卷积核，Padding=2，Stride=2。

**1. 输出特征图尺寸**

通用公式：
$$
H_{out} = \left\lfloor \frac{H_{in} + 2P - K}{S} \right\rfloor + 1
$$

代入数据（H=W=32, P=2, K=5, S=2）：
$$
H_{out} = \left\lfloor \frac{32 + 2\times2 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16
$$

输出通道数等于卷积核个数 = 16。

$$\boxed{\text{输出特征图尺寸：} 16 \times 16 \times 16 \text{（通道数} \times \text{高} \times \text{宽）}}$$

**2. 单个输出像素的乘法次数**

单个卷积核大小为 $3\times5\times5$，计算一个输出像素需要对该感受野内的所有元素做点乘：
$$
\text{乘法次数} = C_{in} \times K_H \times K_W = 3 \times 5 \times 5 = 75
$$

$$\boxed{75 \text{ 次乘法}}$$

### 2.2 编程题：手动实现 Max Pooling

In [1]:
import numpy as np

def max_pool2d(x, kernel_size, stride=1, padding=0):
    """
    手动实现二维最大池化（前向传播）。
    
    参数：
        x: 输入张量，形状为 (N, C, H, W)
        kernel_size: 池化窗口大小（int 或 (kH, kW)）
        stride: 步幅（int 或 (sH, sW)）
        padding: 填充大小（int 或 (pH, pW)）
    
    返回：
        out: 输出张量，形状为 (N, C, H_out, W_out)
    """
    # 处理参数
    if isinstance(kernel_size, int):
        kH, kW = kernel_size, kernel_size
    else:
        kH, kW = kernel_size

    if isinstance(stride, int):
        sH, sW = stride, stride
    else:
        sH, sW = stride

    if isinstance(padding, int):
        pH, pW = padding, padding
    else:
        pH, pW = padding

    N, C, H, W = x.shape

    # 对输入进行零填充
    if pH > 0 or pW > 0:
        x_padded = np.pad(x, ((0, 0), (0, 0), (pH, pH), (pW, pW)),
                          mode='constant', constant_values=-np.inf)
    else:
        x_padded = x

    # 计算输出尺寸
    H_out = (H + 2 * pH - kH) // sH + 1
    W_out = (W + 2 * pW - kW) // sW + 1

    # 初始化输出
    out = np.zeros((N, C, H_out, W_out), dtype=x.dtype)

    # 滑动窗口计算最大值
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * sH
            h_end   = h_start + kH
            w_start = j * sW
            w_end   = w_start + kW
            window = x_padded[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = window.max(axis=(2, 3))

    return out


# ===== 测试 =====
np.random.seed(42)
x = np.random.randn(1, 1, 4, 4).astype(np.float32)
print("输入（1×1×4×4）:")
print(x[0, 0])

# kernel=2, stride=2, padding=0
out = max_pool2d(x, kernel_size=2, stride=2, padding=0)
print("\nMax Pooling 输出（kernel=2, stride=2, padding=0）:")
print(out[0, 0])

# 用 PyTorch 验证
import torch
import torch.nn.functional as F
x_t = torch.from_numpy(x)
out_t = F.max_pool2d(x_t, kernel_size=2, stride=2, padding=0)
print("\nPyTorch 参考输出:")
print(out_t[0, 0].numpy())
print("\n结果一致:", np.allclose(out, out_t.numpy()))

# 测试带 padding 的情况
out_p = max_pool2d(x, kernel_size=3, stride=1, padding=1)
out_p_t = F.max_pool2d(x_t, kernel_size=3, stride=1, padding=1)
print("\n带 padding 结果一致（kernel=3, stride=1, padding=1）:", 
      np.allclose(out_p, out_p_t.numpy()))


输入（1×1×4×4）:
[[ 0.49671414 -0.1382643   0.64768857  1.5230298 ]
 [-0.23415338 -0.23413695  1.5792128   0.7674347 ]
 [-0.46947438  0.54256004 -0.46341768 -0.46572974]
 [ 0.24196227 -1.9132802  -1.7249179  -0.5622875 ]]

Max Pooling 输出（kernel=2, stride=2, padding=0）:
[[ 0.49671414  1.5792128 ]
 [ 0.54256004 -0.46341768]]

PyTorch 参考输出:
[[ 0.49671414  1.5792128 ]
 [ 0.54256004 -0.46341768]]

结果一致: True

带 padding 结果一致（kernel=3, stride=1, padding=1）: True


## 3 LeNet, AlexNet, VGG 和 NiN

### 3.1 理论计算题

**题目：** 输入/输出通道数均为 $C$，比较 5×5 卷积与两个串联 3×3 卷积的参数量。

**1. 单个 5×5 卷积层（不带偏置）的参数量**

$$
\text{params}_{5\times5} = C_{out} \times C_{in} \times K_H \times K_W = C \times C \times 5 \times 5 = 25C^2
$$

$$\boxed{25C^2}$$

**2. 两个串联 3×3 卷积层（不带偏置）的总参数量**

每层参数量 = $C \times C \times 3 \times 3 = 9C^2$，两层合计：

$$
\text{params}_{3\times3 \times 2} = 9C^2 + 9C^2 = 18C^2
$$

$$\boxed{18C^2}$$

**结论：** 两个 3×3 卷积的参数量（$18C^2$）比一个 5×5 卷积（$25C^2$）少 **28%**，同时两层 3×3 卷积拥有**两个 ReLU 非线性激活**，表达能力更强，这是 VGG 用小卷积核替代大卷积核的核心动机。

### 3.2 编程题：实现 NiN Block

In [2]:
import torch
import torch.nn as nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """
    标准 NiN 块（Network in Network Block）。
    
    结构：
        Conv(kernel_size, stride, padding) -> ReLU
        -> Conv(1×1) -> ReLU
        -> Conv(1×1) -> ReLU
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels,
                  kernel_size=kernel_size, stride=stride, padding=padding),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True)
    )


# ===== 测试 =====
block = nin_block(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=0)
print("NiN Block 结构：")
print(block)

# 前向传播验证
x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    y = block(x)
print(f"\n输入形状: {x.shape}")
print(f"输出形状: {y.shape}")


NiN Block 结构：
Sequential(
  (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
  (1): ReLU(inplace=True)
  (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU(inplace=True)
  (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU(inplace=True)
)

输入形状: torch.Size([1, 3, 224, 224])
输出形状: torch.Size([1, 96, 54, 54])


## 4 Inception, 批量归一化和残差网络

### 4.1 理论计算题

**题目：** 4 个样本的特征值 $x_1=2, x_2=4, x_3=6, x_4=8$，$\gamma=2, \beta=1, \varepsilon=0$。

**Batch Normalization 公式：**

$$
\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \varepsilon}}, \quad y_i = \gamma \hat{x}_i + \beta
$$

**步骤一：计算均值**
$$
\mu_B = \frac{2+4+6+8}{4} = \frac{20}{4} = 5
$$

**步骤二：计算方差**
$$
\sigma_B^2 = \frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4} = \frac{9+1+1+9}{4} = \frac{20}{4} = 5
$$

**步骤三：标准化**
$$
\hat{x}_1 = \frac{2-5}{\sqrt{5}} = \frac{-3}{\sqrt{5}}, \quad
\hat{x}_2 = \frac{4-5}{\sqrt{5}} = \frac{-1}{\sqrt{5}}, \quad
\hat{x}_3 = \frac{6-5}{\sqrt{5}} = \frac{1}{\sqrt{5}}, \quad
\hat{x}_4 = \frac{8-5}{\sqrt{5}} = \frac{3}{\sqrt{5}}
$$

**步骤四：缩放平移（$\gamma=2, \beta=1$）**
$$
y_i = 2\hat{x}_i + 1
$$

$$
y_1 = \frac{-6}{\sqrt{5}} + 1 \approx -1.6833, \quad
y_2 = \frac{-2}{\sqrt{5}} + 1 \approx 0.1056
$$
$$
y_3 = \frac{2}{\sqrt{5}} + 1 \approx 1.8944, \quad
y_4 = \frac{6}{\sqrt{5}} + 1 \approx 3.6833
$$

$$\boxed{y_1 \approx -1.6833,\quad y_2 \approx 0.1056,\quad y_3 \approx 1.8944,\quad y_4 \approx 3.6833}$$

In [3]:
import numpy as np

# 验证 BN 计算
x = np.array([2.0, 4.0, 6.0, 8.0])
gamma, beta, eps = 2.0, 1.0, 0.0

mu = x.mean()
sigma2 = x.var()   # 总体方差（BN 使用总体方差）
x_hat = (x - mu) / np.sqrt(sigma2 + eps)
y = gamma * x_hat + beta

print(f"均值 μ = {mu}")
print(f"方差 σ² = {sigma2}")
print(f"标准化后: {x_hat}")
print(f"BN 输出 y = {y}")
print(f"y1={y[0]:.4f}, y2={y[1]:.4f}, y3={y[2]:.4f}, y4={y[3]:.4f}")


均值 μ = 5.0
方差 σ² = 5.0
标准化后: [-1.34164079 -0.4472136   0.4472136   1.34164079]
BN 输出 y = [-1.68328157  0.10557281  1.89442719  3.68328157]
y1=-1.6833, y2=0.1056, y3=1.8944, y4=3.6833


### 4.2 编程题：实现残差块 Residual

In [4]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    残差块（Residual Block）。
    
    结构：
        主路径: Conv3x3 -> BN -> ReLU -> Conv3x3 -> BN
        捷径:   若 use_1x1conv=True，使用 1×1 Conv 调整维度；否则恒等映射
        输出:   F(x) + x 后再经 ReLU
    """
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        # 主路径：两个 3×3 卷积 + BN
        self.conv1 = nn.Conv2d(in_channels, out_channels,
                               kernel_size=3, stride=stride, padding=1)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels,
                               kernel_size=3, stride=1, padding=1)
        self.bn2   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)

        # 捷径连接（可选 1×1 卷积调整通道/尺寸）
        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels,
                                      kernel_size=1, stride=stride)
        else:
            self.shortcut = None

    def forward(self, x):
        # 主路径
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # 捷径
        identity = self.shortcut(x) if self.shortcut else x
        # 残差相加后激活
        return self.relu(out + identity)


# ===== 测试1：通道数不变，无 1×1 卷积 =====
blk1 = Residual(64, 64, use_1x1conv=False)
x1 = torch.randn(2, 64, 16, 16)
with torch.no_grad():
    y1 = blk1(x1)
print(f"测试1（无1×1卷积）- 输入: {x1.shape}, 输出: {y1.shape}")

# ===== 测试2：通道数翻倍，使用 1×1 卷积 =====
blk2 = Residual(64, 128, use_1x1conv=True, stride=2)
x2 = torch.randn(2, 64, 16, 16)
with torch.no_grad():
    y2 = blk2(x2)
print(f"测试2（有1×1卷积，步幅=2）- 输入: {x2.shape}, 输出: {y2.shape}")

print("\n残差块定义：")
print(blk1)


测试1（无1×1卷积）- 输入: torch.Size([2, 64, 16, 16]), 输出: torch.Size([2, 64, 16, 16])
测试2（有1×1卷积，步幅=2）- 输入: torch.Size([2, 64, 16, 16]), 输出: torch.Size([2, 128, 8, 8])

残差块定义：
Residual(
  (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
)


## 5 图像增广，微调和样式迁移

### 5.1 理论计算题

**问题 1：为什么底层特征提取层用小学习率、顶层输出层用大学习率？**

预训练模型的**底层特征提取层**（如卷积层前几层）已经学到了具有通用性的特征表示（边缘、纹理等），这些特征对新任务同样有效。若设置过大的学习率，会破坏这些已经优化好的参数，导致"灾难性遗忘"（catastrophic forgetting）。因此：

- **底层层使用小学习率（甚至冻结）**：保留预训练到的通用特征，防止过拟合，加快收敛；
- **顶层输出层使用大学习率**：顶层是针对新任务随机初始化的，需要快速调整以适应新的类别分布；参数从零开始，需要更大的更新步长。

这种差异化学习率策略的核心思想是：**底层特征可复用，顶层需重新学习**。

---

**问题 2：目标数据集非常小且与源数据集非常相似时的微调策略**

当目标数据集**非常小**且**与源数据集非常相似**时，过拟合风险极高，推荐策略：

1. **冻结大部分（甚至全部）底层参数**：只更新顶层的输出层，因为底层特征已与目标任务高度匹配；
2. **使用非常小的全局学习率**：防止有限数据导致模型过拟合；
3. **增加正则化手段**：如 Dropout、权重衰减（L2 正则）；
4. **使用数据增广**：人工扩充小数据集，提升泛化能力；
5. **极端情况下**：将预训练模型完全冻结，仅将其作为特征提取器，只训练最后一个线性分类层（Linear Probe）。

### 5.2 编程题：图像增广 Pipeline

In [5]:
import os, getpass
# Windows 环境下 torchvision 依赖 USERNAME 环境变量
if os.name == 'nt' and 'USERNAME' not in os.environ:
    os.environ['USERNAME'] = 'user'
import torchvision.transforms as transforms

# 组合图像增广管道
train_augs = transforms.Compose([
    # 1. 随机裁剪：面积比例 [0.08, 1.0]，并缩放到 224×224
    transforms.RandomResizedCrop(
        size=224,
        scale=(0.08, 1.0)
    ),
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 随机改变亮度、对比度、饱和度，变化范围 0.5
    transforms.ColorJitter(
        brightness=0.5,
        contrast=0.5,
        saturation=0.5
    ),
    # 4. 转换为 PyTorch 张量
    transforms.ToTensor()
])

print("图像增广 Pipeline：")
for i, t in enumerate(train_augs.transforms):
    print(f"  {i+1}. {t}")

# ===== 功能验证（用随机 NumPy 图像模拟 PIL Image）=====
from PIL import Image
import numpy as np

dummy_img = Image.fromarray(
    np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8)
)
aug_tensor = train_augs(dummy_img)
print(f"\n输入图像尺寸: {dummy_img.size}")
print(f"增广后张量形状: {aug_tensor.shape}")   # 期望: torch.Size([3, 224, 224])
print(f"张量值域: [{aug_tensor.min():.3f}, {aug_tensor.max():.3f}]")


图像增广 Pipeline：
  1. RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
  2. RandomHorizontalFlip(p=0.5)
  3. ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
  4. ToTensor()

输入图像尺寸: (256, 256)
增广后张量形状: torch.Size([3, 224, 224])
张量值域: [0.231, 0.659]


## 6 目标检测，计算机视觉训练技巧

### 6.1 理论计算题

**题目：** 真实框 $A=[10,10,50,50]$，预测框 $B=[30,30,70,70]$，计算 IoU。

**步骤一：计算交集矩形坐标**

$$
x_1^{inter} = \max(10, 30) = 30, \quad y_1^{inter} = \max(10, 30) = 30
$$
$$
x_2^{inter} = \min(50, 70) = 50, \quad y_2^{inter} = \min(50, 70) = 50
$$

交集宽度 = $50 - 30 = 20$，交集高度 = $50 - 30 = 20$。

**步骤二：计算交集面积**
$$
S_{inter} = 20 \times 20 = 400
$$

**步骤三：计算各框面积**
$$
S_A = (50-10) \times (50-10) = 40 \times 40 = 1600
$$
$$
S_B = (70-30) \times (70-30) = 40 \times 40 = 1600
$$

**步骤四：计算并集面积**
$$
S_{union} = S_A + S_B - S_{inter} = 1600 + 1600 - 400 = 2800
$$

**步骤五：计算 IoU**
$$
\text{IoU} = \frac{S_{inter}}{S_{union}} = \frac{400}{2800} = \frac{1}{7} \approx 0.1429
$$

$$\boxed{\text{IoU} = \dfrac{1}{7} \approx 0.1429}$$

In [6]:
import numpy as np

def compute_iou(box_a, box_b):
    """
    计算两个边界框的 IoU。
    box 格式: [x1, y1, x2, y2]（左上角, 右下角）
    """
    # 交集坐标
    inter_x1 = max(box_a[0], box_b[0])
    inter_y1 = max(box_a[1], box_b[1])
    inter_x2 = min(box_a[2], box_b[2])
    inter_y2 = min(box_a[3], box_b[3])

    # 交集面积（无交集则为 0）
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    # 各框面积
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])

    # 并集面积
    union_area = area_a + area_b - inter_area

    iou = inter_area / union_area if union_area > 0 else 0.0
    return iou


A = [10, 10, 50, 50]
B = [30, 30, 70, 70]
iou = compute_iou(A, B)
print(f"真实框 A = {A}")
print(f"预测框 B = {B}")
print(f"交集面积 = {(min(50,70)-max(10,30)) * (min(50,70)-max(10,30))}")
print(f"并集面积 = {40*40 + 40*40 - 20*20}")
print(f"IoU = {iou:.6f} = 1/7 ≈ {1/7:.6f}")


真实框 A = [10, 10, 50, 50]
预测框 B = [30, 30, 70, 70]
交集面积 = 400
并集面积 = 2800
IoU = 0.142857 = 1/7 ≈ 0.142857


### 6.2 编程题：标签平滑交叉熵损失

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, targets, epsilon=0.1, reduction='mean'):
    """
    标签平滑交叉熵损失。
    
    参数：
        logits:   模型原始输出，形状 (N, K)
        targets:  真实标签索引，形状 (N,)
        epsilon:  平滑因子，默认 0.1
        reduction: 'mean' | 'sum' | 'none'
    
    原理：
        - 真实类别目标概率：1 - epsilon
        - 其余每个类别目标概率：epsilon / (K - 1)
    """
    N, K = logits.shape

    # 计算 log softmax
    log_probs = F.log_softmax(logits, dim=-1)   # (N, K)

    # 构建平滑后的软标签分布
    # 初始化为 epsilon / (K-1) 的均匀分布
    smooth_labels = torch.full_like(log_probs, epsilon / (K - 1))
    # 真实类别处填入 1 - epsilon
    smooth_labels.scatter_(1, targets.unsqueeze(1), 1.0 - epsilon)

    # 交叉熵 = -sum(smooth_labels * log_probs)
    loss = -(smooth_labels * log_probs).sum(dim=-1)   # (N,)

    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:
        return loss


# ===== 测试 =====
torch.manual_seed(0)
K = 5       # 类别数
N = 4       # batch size
epsilon = 0.1

logits  = torch.randn(N, K)
targets = torch.randint(0, K, (N,))

loss = label_smoothing_cross_entropy(logits, targets, epsilon=epsilon)
print(f"类别数 K={K}, 批大小 N={N}, epsilon={epsilon}")
print(f"logits:\n{logits}")
print(f"targets: {targets.tolist()}")
print(f"\n标签平滑交叉熵损失: {loss.item():.4f}")

# ===== 与 PyTorch 内置实现对比 =====
# 注意：PyTorch nn.CrossEntropyLoss(label_smoothing=ε) 使用 ε/K 均匀分布，
# 本实现按题目要求使用 ε/(K-1) 分布（真实类目标=1-ε，其余各=ε/(K-1)），
# 两者在数值上略有差异，均为有效的标签平滑变体。
ce_smooth = nn.CrossEntropyLoss(label_smoothing=epsilon)
loss_ref = ce_smooth(logits, targets)
print(f"本实现（题目定义，其余类各 ε/(K-1)）: {loss.item():.4f}")
print(f"PyTorch 内置（其余类各 ε/K）:         {loss_ref.item():.4f}")
print(f"（两者分布定义不同，数值略有差异，属正常现象）")

# ===== 验证软标签分布 =====
print(f"\n--- 软标签验证（K={K}, epsilon={epsilon}）---")
print(f"真实类别目标概率: {1 - epsilon}")
print(f"其余类别目标概率: {epsilon / (K - 1):.4f} (每个，共 K-1={K-1} 个)")
print(f"概率之和: {(1 - epsilon) + (K - 1) * epsilon / (K - 1)}")


类别数 K=5, 批大小 N=4, epsilon=0.1
logits:
tensor([[-1.1258, -1.1524, -0.2506, -0.4339,  0.5988],
        [-1.5551, -0.3414,  1.8530,  0.4681, -0.1577],
        [ 1.4437,  0.2660,  1.3894,  1.5863,  0.9463],
        [-0.8437,  0.9318,  1.2590,  2.0050,  0.0537]])
targets: [2, 4, 1, 3]

标签平滑交叉熵损失: 1.8430
本实现（题目定义，其余类各 ε/(K-1)）: 1.8430
PyTorch 内置（其余类各 ε/K）:         1.8400
（两者分布定义不同，数值略有差异，属正常现象）

--- 软标签验证（K=5, epsilon=0.1）---
真实类别目标概率: 0.9
其余类别目标概率: 0.0250 (每个，共 K-1=4 个)
概率之和: 1.0
